# Notebook 7: network sparsity and stage robustness

Conjugate out-of-sample robustness across observed-network sparsity and admissible stage depth.

In [1]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import numpy as np
import pandas as pd

QUICK = quick_mode()
selection = load_result("nb0_selection")["selection"]
P = int(selection["p"])

d = load_data(network="geographic")
Y_full = d["Y_full"].to_numpy()
Y_train = d["Y_train"].to_numpy()
test_start = d["test_start"]
Y_future = Y_full[test_start:]
test_dates = pd.to_datetime(d["test"].index)
N = d["N"]
networks = d["networks"]

NETWORKS = ["geographic", "export", "import"]
K_VALUES = [1, 2, None] if QUICK else list(range(1, N - 1)) + [None]
PRIOR = dict(
    prior_type="minnesota",
    lambda1=MINN["lambda1"], lambda2=MINN["lambda2"],
    lambda3=MINN["lambda3"], rw_centre=MINN["rw_centre"],
)

def graph_summary(W):
    A = np.asarray(W) > 0
    np.fill_diagonal(A, False)
    sw = compute_stage_weights(W, 2)
    return {
        "density": float(A.sum() / (N * (N - 1))),
        "stage2_pairs": int((sw[1] > 0).sum()),
    }

def conjugate_score(panel, target, W, max_stage):
    stages = [max_stage] * P if max_stage else [0] * P
    sw = compute_stage_weights(W, max_stage)
    model = BayesianGNAR(p=P, s=stages, **PRIOR)
    model.fit(panel[:test_start], sw)
    mean, var = model.forecast_test(panel, test_start, len(target))
    score = score_forecasts(target, mean, var, test_dates)
    return model, score, crps_series(target, mean, var)

## Raw-panel sparsity grid

In [2]:
ar_model, ar_score, ar_loss = conjugate_score(Y_full, Y_future, networks["geographic"], 0)

raw_rows = []
for network in NETWORKS:
    for k in K_VALUES:
        Wk = knn_sparsify(networks[network], k)
        g = graph_summary(Wk)
        k_label = "complete" if k is None else str(k)

        for max_stage in [1, 2]:
            if max_stage == 2 and g["stage2_pairs"] == 0:
                continue
            model, score, loss = conjugate_score(Y_full, Y_future, Wk, max_stage)
            X, _, names = build_design(
                Y_train, Wk, p=P, stages=[max_stage] * P,
                mode="global_gnar", h=1,
            )
            vif1, r21 = variance_inflation_factor(X, names.index("net_lag1_stage1"))
            if max_stage == 2:
                vif2, r22 = variance_inflation_factor(X, names.index("net_lag1_stage2"))
            else:
                vif2 = r22 = np.nan

            raw_rows.append({
                "network": network,
                "k": k_label,
                "density": g["density"],
                "stage2_pairs": g["stage2_pairs"],
                "max_stage": max_stage,
                "CRPS": score["CRPS"],
                "RMSE": score["RMSE"],
                "gain_vs_AR": ar_score["CRPS"] - score["CRPS"],
                "log_marginal_likelihood": model.log_marginal_likelihood(),
                "stage1_VIF": vif1, "stage1_R2": r21,
                "stage2_VIF": vif2, "stage2_R2": r22,
            })

raw_grid = pd.DataFrame(raw_rows)
print(raw_grid.round(6).to_string(index=False))

   network        k  density  stage2_pairs  max_stage     CRPS     RMSE  gain_vs_AR  log_marginal_likelihood  stage1_VIF  stage1_R2  stage2_VIF  stage2_R2
geographic        1 0.045455            11          1 0.259192 0.503759    0.003257             -2508.375074   64.693967   0.984543         NaN        NaN
geographic        1 0.045455            11          2 0.258733 0.503034    0.003716             -2504.252127   66.401216   0.984940  104.589034   0.990439
geographic        2 0.090909            34          1 0.257880 0.501144    0.004569             -2503.595915   96.235078   0.989609         NaN        NaN
geographic        2 0.090909            34          2 0.257213 0.500080    0.005236             -2503.172583  104.360103   0.990418   71.710909   0.986055
geographic        3 0.136364            64          1 0.257770 0.501269    0.004679             -2504.532766   83.223434   0.987984         NaN        NaN
geographic        3 0.136364            64          2 0.256460 0.49925

## Stage order on raw and PC1-adjusted panels

In [3]:
mu_tr = Y_train.mean(axis=0, keepdims=True)
_, _, Vt = np.linalg.svd(Y_train - mu_tr, full_matrices=False)
load1 = Vt[0]
centred = Y_full - mu_tr
Y_resid = centred - np.outer(centred @ load1, load1)
Y_resid_future = Y_resid[test_start:]

def stage_pairs_from_grid(grid):
    rows = []
    for (network, k), sub in grid.groupby(["network", "k"]):
        if set(sub["max_stage"]) != {1, 2}:
            continue
        s1 = sub[sub["max_stage"] == 1].iloc[0]
        s2 = sub[sub["max_stage"] == 2].iloc[0]
        rows.append({
            "network": network, "k": k, "density": s1["density"],
            "stage2_pairs": int(s1["stage2_pairs"]),
            "stage1_CRPS": s1["CRPS"], "stage2_CRPS": s2["CRPS"],
            "stage2_minus_stage1_CRPS": s2["CRPS"] - s1["CRPS"],
            "stage1_log_marginal_likelihood": s1["log_marginal_likelihood"],
            "stage2_log_marginal_likelihood": s2["log_marginal_likelihood"],
            "stage2_minus_stage1_log_marginal_likelihood":
                s2["log_marginal_likelihood"] - s1["log_marginal_likelihood"],
        })
    return pd.DataFrame(rows)

stage_order_raw = stage_pairs_from_grid(raw_grid)

resid_rows = []
for network in NETWORKS:
    for k in K_VALUES:
        Wk = knn_sparsify(networks[network], k)
        g = graph_summary(Wk)
        if g["stage2_pairs"] == 0:
            continue
        k_label = "complete" if k is None else str(k)
        for max_stage in [1, 2]:
            model, score, _ = conjugate_score(Y_resid, Y_resid_future, Wk, max_stage)
            resid_rows.append({
                "network": network, "k": k_label, "density": g["density"],
                "stage2_pairs": g["stage2_pairs"], "max_stage": max_stage,
                "CRPS": score["CRPS"], "RMSE": score["RMSE"],
                "log_marginal_likelihood": model.log_marginal_likelihood(),
            })

resid_grid = pd.DataFrame(resid_rows)
stage_order_residual = stage_pairs_from_grid(resid_grid)

print(f"raw admissible stage-2 cells: {len(stage_order_raw)}")
print(f"residual admissible stage-2 cells: {len(stage_order_residual)}")

raw admissible stage-2 cells: 63
residual admissible stage-2 cells: 63


In [4]:
save_result("nb7_network_robustness", {
    "p": P,
    "k_values": ["complete" if k is None else k for k in K_VALUES],
    "ar_baseline": {"CRPS": ar_score["CRPS"], "RMSE": ar_score["RMSE"]},
    "raw_grid": raw_grid.to_dict(orient="records"),
    "stage_order_raw": stage_order_raw.to_dict(orient="records"),
    "residual_grid": resid_grid.to_dict(orient="records"),
    "stage_order_residual": stage_order_residual.to_dict(orient="records"),
    "config": run_config(p=P, stages=[2] * P, purpose="network_robustness"),
    "quick": QUICK,
})
print("saved nb7_network_robustness")

saved nb7_network_robustness
